# Baseline YOLO26n — AMD cloud / ROCm + Roboflow

Clone standalone baseline 03, tanpa teacher/KD. Protokol baseline tetap: 100 epoch per seed,
640px, batch16, AdamW lr0.001, AMP True. Target Linux dengan image PyTorch ROCm yang sudah siap.
Notebook tidak memasang driver/ROCm atau mengganti torch/torchvision image.

**Cara menjalankan:** pilih kernel image ROCm, isi kedua API key pada Section 1, periksa
konfigurasi lalu Run All. Dataset diunduh/divalidasi satu kali. Seed **7 lalu 3407** dijalankan
serial dengan model, run W&B dan direktori terpisah. Seed 42/123/2024 ditampilkan sebagai komentar.

Validasi dan pemilihan best checkpoint memakai **val**; test tidak dipakai untuk tuning.
Dataset Roboflow belum dibuktikan identik dengan `poltrada-v3-yolo26-clean` Kaggle; hasil lama
tidak digabung otomatis. Lingkungan AMD, identitas Roboflow, hash YAML dan kelas dicatat.
Hardware AMD/cloud dan API nyata belum diuji dari lingkungan lokal; tes sintetis tidak mengukur
performa Poltrada. Tidak ada auto-resume; setelah gagal, isi hanya seed yang belum selesai.


## 1. Kredensial API — isi manual

Isi variabel di cell berikut. Nilai tidak dicetak/disertakan ke config atau artifact; penyimpanan kode otomatis W&B dinonaktifkan. Tidak ada prompt getpass atau pembacaan key dari environment.


In [ ]:
# Isi sendiri sebelum Run All. Kedua nilai hanya dipakai untuk autentikasi.
WANDB_API_KEY = ""
ROBOFLOW_API_KEY = ""


def validate_api_keys():
    for name, value in (("WANDB_API_KEY", WANDB_API_KEY), ("ROBOFLOW_API_KEY", ROBOFLOW_API_KEY)):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"Isi {name} pada cell kredensial sebelum melanjutkan.")


validate_api_keys()


## 2. Konfigurasi penelitian dan sumber data

Seed aktif dapat diatur dengan mengomentari/mengaktifkan elemen list. Seluruh seed aktif berjalan serial. Path memakai home pengguna cloud; tidak ada ketergantungan Kaggle.


In [ ]:
from pathlib import Path

EXPERIMENT_ID = "03_baseline_yolo26n_amd"
METHOD = "none"
MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEEDS_TO_RUN = [
    # 42,    # Sudah selesai
    # 123,   # Sudah selesai
    # 2024,  # Sudah selesai
    7,
    3407,
]
SEED = SEEDS_TO_RUN[0] if SEEDS_TO_RUN else None
DEVICE = 0  # AMD ROCm menggunakan API torch.cuda dan indeks GPU yang sama.
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"
PROJECT = str(Path.home() / "skripsi-kd/results")
TEACHER_CHECKPOINT = None

ROBOFLOW_WORKSPACE = "ilham-rafiqin"
ROBOFLOW_PROJECT = "poltrada-original-raw-2"
ROBOFLOW_VERSION = 3
ROBOFLOW_FORMAT = "yolo26"
DATASET_DIR = Path.home() / "skripsi-kd/datasets/poltrada-v3"
DATA = None  # Diisi dari Path(dataset.location) / "data.yaml" setelah download/validasi.

WANDB_ENABLED = True
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_ENTITY = None
WANDB_GROUP = "supervised"
WANDB_TAGS = ["baseline", "model-n", f"seed{SEED}", "poltrada-v3",
              "amd-rocm", f"roboflow-v{ROBOFLOW_VERSION}", EXPERIMENT_ID]
BASE_WANDB_TAGS = [tag for tag in WANDB_TAGS if not tag.startswith("seed")]


## 3. Pemeriksaan AMD dan dependensi

Image harus sudah menyediakan PyTorch ROCm dan torchvision yang kompatibel serta akses GPU.
`torch.version.hip` membedakan ROCm dari build CUDA/CPU. Pada AMD tetap gunakan `DEVICE=0`
dan `torch.cuda` ([PyTorch HIP semantics](https://docs.pytorch.org/docs/stable/notes/hip.html)).
Notebook tidak mengubah GPU visibility, driver atau runtime image. Jika GPU tidak dapat diakses,
perbaiki image/akses perangkat cloud. Sesudah paket yang sudah di-import diperbarui, restart kernel.

Untuk server tanpa desktop, gunakan distribusi resmi `ultralytics-opencv-headless==8.4.155`
dan `opencv-python-headless==4.10.0.84` (sesuai pin Roboflow 1.4.0). Installer membersihkan
provider desktop/namespace yang bertumpuk, mempertahankan torch/torchvision ROCm, kemudian
menguji import cv2, GUI=NONE, PNG dan resize di subprocess sebelum import Roboflow/Ultralytics.
Ini menghindari ketergantungan GUI seperti libxcb.so.1/libGL.so.1 tanpa memasang paket sistem.
Lihat [panduan headless Ultralytics](https://docs.ultralytics.com/quickstart#headless-server-installation)
dan [instalasi PyTorch AMD](https://rocm.docs.amd.com/projects/ai-ecosystem/en/latest/frameworks/pytorch/install.html).


In [ ]:
import gc
import hashlib
import io
import json
import os
import platform
import subprocess
import sys
import time
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from datetime import datetime, timezone
from importlib import metadata
from tempfile import TemporaryDirectory

import torch
import torchvision


def require_rocm_runtime():
    """Reject CPU/CUDA builds and unavailable HIP devices; probe the selected AMD GPU."""
    if platform.system() != "Linux":
        raise RuntimeError("Notebook ini memerlukan Linux cloud dengan PyTorch ROCm siap pakai.")
    if not torch.version.hip:
        raise RuntimeError("PyTorch bukan build ROCm/HIP. Pilih image PyTorch ROCm, bukan CUDA/CPU.")
    if DEVICE != 0 or isinstance(DEVICE, bool) or not torch.cuda.is_available():
        raise RuntimeError("GPU AMD tidak tersedia; gunakan DEVICE=0 dan periksa akses GPU image cloud.")
    torch.cuda.set_device(DEVICE)
    probe = torch.ones((8, 8), device=f"cuda:{DEVICE}")
    try:
        product = probe @ probe
        if not bool(torch.isfinite(product).all().item()):
            raise RuntimeError("Operasi tensor GPU menghasilkan nilai non-finite.")
        torch.cuda.synchronize(DEVICE)
    finally:
        del probe
    properties = torch.cuda.get_device_properties(DEVICE)
    return {"backend": "rocm", "hip": str(torch.version.hip),
            "gpu": torch.cuda.get_device_name(DEVICE),
            "gpu_architecture": getattr(properties, "gcnArchName", None),
            "total_vram_bytes": int(properties.total_memory)}


def install_research_packages():
    """Use one headless cv2/Ultralytics provider and preserve the image's ROCm stack."""
    pinned_stack = {name: metadata.version(name) for name in ("torch", "torchvision")}
    wanted = {"ultralytics-opencv-headless": "8.4.155", "opencv-python-headless": "4.10.0.84",
              "wandb": "0.28.1", "roboflow": "1.4.0"}

    def version_or_none(name):
        try:
            return metadata.version(name)
        except metadata.PackageNotFoundError:
            return None

    numpy_before = version_or_none("numpy")
    cv_packages = ("opencv-python", "opencv-contrib-python", "opencv-contrib-python-headless", "opencv-python-headless")
    remove = []
    # Packages share the same import namespace. Remove every provider in the conflicted family,
    # then reinstall the headless provider; otherwise pip may leave a partly deleted cv2 module.
    if version_or_none("ultralytics") is not None:
        remove.extend(name for name in ("ultralytics", "ultralytics-opencv-headless") if version_or_none(name) is not None)
    if any(version_or_none(name) is not None for name in cv_packages[:-1]):
        remove.extend(name for name in cv_packages if version_or_none(name) is not None)
    pending_names = [name for name, version in wanted.items() if version_or_none(name) != version or name in remove]
    affected = {"ultralytics-opencv-headless": "ultralytics", "opencv-python-headless": "cv2",
                "wandb": "wandb", "roboflow": "roboflow"}
    imported = any(module == affected[name] or module.startswith(affected[name] + ".")
                   for name in pending_names for module in sys.modules)
    if remove and any(module == "roboflow" or module.startswith("roboflow.") for module in sys.modules):
        imported = True
    if pending_names or remove:
        with TemporaryDirectory(prefix="baseline_rocm_") as temporary:
            constraints = Path(temporary) / "rocm-stack.txt"
            constraints.write_text("".join(f"{name}=={version}\n" for name, version in pinned_stack.items()), encoding="utf-8")
            print("Menyiapkan Ultralytics/OpenCV headless; versi torch/torchvision image dikunci.", flush=True)
            if remove:
                subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", *remove])
            if pending_names:
                pending = [f"{name}=={wanted[name]}" for name in pending_names]
                subprocess.check_call([sys.executable, "-m", "pip", "install", "--constraint", str(constraints), *pending])
    if any(metadata.version(name) != version for name, version in pinned_stack.items()):
        raise RuntimeError("Stack torch/torchvision berubah; pulihkan image ROCm sebelum training.")
    if any(version_or_none(name) != version for name, version in wanted.items()):
        raise RuntimeError("Versi paket penelitian belum sesuai konfigurasi.")
    if version_or_none("ultralytics") is not None or any(version_or_none(name) is not None for name in cv_packages[:-1]):
        raise RuntimeError("Provider OpenCV/Ultralytics desktop masih terpasang; jangan campur namespace cv2/ultralytics.")
    if version_or_none("numpy") != numpy_before and "numpy" in sys.modules:
        imported = True  # torchvision may have imported NumPy before pip resolved Roboflow's numpy<2.4.
    opencv_info = verify_headless_opencv()
    if imported:
        raise RuntimeError("Paket yang sudah di-import telah diperbarui. Restart kernel dan Run All kembali.")
    return opencv_info


def verify_headless_opencv():
    # Fresh subprocess also detects missing native libraries and stale/overlapping cv2 files
    # without importing a potentially broken native extension into this kernel.
    script = """import json, re
import cv2
import numpy as np
assert cv2.__version__ == '4.10.0', 'Versi modul cv2 tidak sesuai distribusi yang dipin'
assert re.search(r'^\\s*GUI:\\s+NONE\\s*$', cv2.getBuildInformation(), re.MULTILINE), 'cv2 bukan build headless'
image = np.zeros((4, 4, 3), dtype=np.uint8)
success, encoded = cv2.imencode('.png', image)
assert success
assert cv2.imdecode(encoded, cv2.IMREAD_COLOR).shape == image.shape
assert cv2.resize(image, (2, 2)).shape == (2, 2, 3)
print(json.dumps({'opencv': cv2.__version__, 'opencv_gui': 'NONE'}))
"""
    probe = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, check=False)
    if probe.returncode:
        raise RuntimeError("OpenCV headless gagal di subprocess; training belum dimulai.\n" + probe.stderr[-1500:])
    try:
        return json.loads(probe.stdout)
    except ValueError:
        raise RuntimeError("Output pemeriksaan OpenCV headless tidak valid; training belum dimulai.") from None


validate_api_keys()
AMD_RUNTIME_INFO = require_rocm_runtime()
OPENCV_RUNTIME_INFO = install_research_packages()

from roboflow import Roboflow
from ultralytics import YOLO, settings
from ultralytics import __version__ as ULTRALYTICS_VERSION
from ultralytics.data.utils import IMG_FORMATS, check_det_dataset
from ultralytics.utils.torch_utils import init_seeds

import wandb

settings.update({"wandb": False})  # Gunakan callback notebook saja untuk logging.
wandb.login(key=WANDB_API_KEY)
ENVIRONMENT_INFO = {
    "python": sys.version, "platform": platform.platform(),
    "torch": str(torch.__version__), "torchvision": str(torchvision.__version__),
    "ultralytics": ULTRALYTICS_VERSION, "ultralytics_distribution": "ultralytics-opencv-headless",
    "opencv_distribution": metadata.version("opencv-python-headless"), "numpy": metadata.version("numpy"),
    "wandb": metadata.version("wandb"),
    "roboflow": metadata.version("roboflow"), **AMD_RUNTIME_INFO, **OPENCV_RUNTIME_INFO,
}
print(json.dumps(ENVIRONMENT_INFO, indent=2))


## 4. Download dan validasi Roboflow

Workspace/project/version/format/lokasi berasal dari konfigurasi. SDK memakai key dari Section 1.
Dataset diunduh sekali dan digunakan kedua seed. Folder cache yang tidak lengkap atau metadata
versinya berbeda menghentikan proses; tidak ditimpa/dihapus otomatis. Path relatif YAML
diselesaikan loader resmi. Tidak ada cleaning, perubahan label/kelas, augmentasi tambahan atau resplit.


In [ ]:
def class_names(names):
    if isinstance(names, dict):
        return [str(names[key]) for key in sorted(names, key=lambda value: int(value))]
    return [str(value) for value in names]


def validate_roboflow_dataset(yaml_path):
    yaml_path = Path(yaml_path).resolve()
    if not yaml_path.is_file():
        raise FileNotFoundError(f"Cache/download belum lengkap: data.yaml tidak tersedia di {yaml_path.parent}.")
    info = check_det_dataset(str(yaml_path), autodownload=False)
    if int(info["nc"]) <= 0 or not info.get("train") or not info.get("val"):
        raise ValueError("Dataset deteksi harus memiliki kelas serta split train dan val.")
    for split in ("train", "val", "test"):
        for raw_path in ([info[split]] if isinstance(info.get(split), str) else info.get(split) or []):
            path = Path(raw_path).resolve()
            if not path.exists() or not path.is_relative_to(yaml_path.parent):
                raise FileNotFoundError(f"Split {split} tidak tersedia di dalam direktori dataset: {path}")
            if path.is_dir() and not any(file.is_file() and file.suffix[1:].lower() in IMG_FORMATS for file in path.rglob("*")):
                raise FileNotFoundError(f"Cache/download belum lengkap: split {split} tidak berisi gambar.")
    declared = info.get("roboflow") or {}
    for name, expected in (("workspace", ROBOFLOW_WORKSPACE), ("project", ROBOFLOW_PROJECT), ("version", ROBOFLOW_VERSION)):
        if name in declared and str(declared[name]) != str(expected):
            raise ValueError(f"Metadata Roboflow {name} berbeda dari konfigurasi; gunakan direktori dataset yang sesuai.")
    return info


def download_roboflow_dataset():
    validate_api_keys()
    destination = Path(DATASET_DIR).expanduser().resolve()
    cache_reused = destination.exists()
    if cache_reused:
        # Jangan biarkan SDK menganggap folder kosong/parsial sebagai download sukses.
        validate_roboflow_dataset(destination / "data.yaml")
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Dataset Roboflow {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}, format={ROBOFLOW_FORMAT}...", flush=True)
    # SDK dapat mencetak URL download bertoken saat error; jangan teruskan output/raw exception tersebut.
    captured = io.StringIO()
    try:
        with redirect_stdout(captured), redirect_stderr(captured):
            rf = Roboflow(api_key=ROBOFLOW_API_KEY)
            project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
            dataset = project.version(ROBOFLOW_VERSION).download(ROBOFLOW_FORMAT, location=str(destination))
    except Exception as error:  # noqa: BLE001 -- SDK errors/URLs may contain the API key.
        raise RuntimeError(f"Download Roboflow gagal ({type(error).__name__}). Periksa key, akses project, format dan koneksi; cache parsial tidak ditimpa otomatis.") from None
    finally:
        captured.close()
    yaml_path = Path(dataset.location).resolve() / "data.yaml"
    if yaml_path.parent != destination:
        raise ValueError("Lokasi download SDK berbeda dari DATASET_DIR.")
    info = validate_roboflow_dataset(yaml_path)
    provenance = {
        "source": "roboflow", "workspace": ROBOFLOW_WORKSPACE, "project": ROBOFLOW_PROJECT,
        "version": ROBOFLOW_VERSION, "format": ROBOFLOW_FORMAT, "cache_reused": cache_reused,
        "data_yaml": str(yaml_path), "yaml_sha256": hashlib.sha256(yaml_path.read_bytes()).hexdigest(),
        "nc": int(info["nc"]), "class_names": class_names(info["names"]),
        "splits": {name: info[name] for name in ("train", "val", "test") if info.get(name)},
        "kaggle_dataset_equivalence": "NOT_VERIFIED",
    }
    return str(yaml_path), info, provenance


DATA, data_info, DATASET_PROVENANCE = download_roboflow_dataset()
print("Dataset:", DATA)
print("Classes:", DATASET_PROVENANCE["class_names"])


## 5. Baseline dan pengaman protokol

Student baru dimuat dari pretrained per seed. Loss deteksi bawaan Ultralytics tetap dipakai; tidak ada distillation. AMP efektif, batch dan resolusi harus sama dengan konfigurasi. Jika check AMP ROCm menonaktifkannya, run dihentikan; perubahan AMP harus eksplisit, bukan fallback otomatis.


In [ ]:
def log_status(stage, message):
    stamp = datetime.now().astimezone().strftime("%H:%M:%S")
    print(f"[{stamp}] [{stage}] {message}", flush=True)

print("Student baru akan dibuat terpisah untuk setiap seed.")


In [ ]:
KD_TRAINER = None
KD_TRAIN_ARGS = {}
KD_CONFIG = {}

In [ ]:
def require_training_device(device):
    if not torch.version.hip or device.type != "cuda" or device.index != DEVICE:
        raise RuntimeError("Training harus memakai GPU AMD ROCm pada DEVICE=0; tidak ada fallback CPU/CUDA NVIDIA.")


def strict_training_protocol(trainer):
    require_training_device(trainer.device)
    if trainer.batch_size != BATCH or trainer.args.batch != BATCH or trainer.args.imgsz != IMGSZ:
        raise RuntimeError("Batch/resolusi efektif berubah dari konfigurasi; antrean dihentikan.")
    if bool(trainer.amp) != AMP:
        raise RuntimeError("AMP efektif berbeda dari konfigurasi. Periksa dukungan ROCm; perubahan AMP harus dilakukan eksplisit pada konfigurasi.")


def strict_batch_start(trainer):
    strict_training_protocol(trainer)
    trainer._oom_retries = 3  # Engine 8.4.155: langsung raise; tidak mengurangi batch saat OOM.


def strict_batch_end(trainer):
    trainer._oom_retries = 0
    if not torch.isfinite(trainer.loss).all():
        raise RuntimeError("Loss non-finite; hentikan run tanpa mengubah protokol atau mengulang seed otomatis.")


## 6. Training dan antrean seed

Cell berikut mendefinisikan fungsi; training dimulai di Section 8. Satu run W&B baru per seed, group supervised, tag amd-rocm/roboflow-v3. Checkpoint dan metrik diunggah per epoch; upload ditunggu sebelum lanjut. Kegagalan menutup run dan menghentikan antrean; hasil seed sebelumnya tetap disimpan.


In [ ]:
@contextmanager
def exclusive_gpu():
    lock = Path(PROJECT).parent / ".baseline_yolo26n_amd_gpu.lock"
    lock.parent.mkdir(parents=True, exist_ok=True)
    try:
        descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError as error:
        raise RuntimeError(f"GPU dikunci eksperimen lain: {lock}.") from error
    try:
        with os.fdopen(descriptor, "w") as stream:
            stream.write(json.dumps({"pid": os.getpid(), "experiment": EXPERIMENT_ID, "run": wb_run.id}))
        yield
    finally:
        lock.unlink(missing_ok=True)


def prepare_seed_teacher():
    # Baseline YOLO26n tidak menggunakan teacher.
    pass

def save_epoch_artifact(trainer):
    """Snapshot checkpoint setiap epoch; upload selesai sebelum last.pt ditimpa epoch berikutnya."""
    epoch = int(trainer.epoch) + 1
    last_path = Path(trainer.last)
    best_path = Path(trainer.best)
    if not last_path.is_file():
        raise RuntimeError(f"Checkpoint epoch {epoch} tidak ada: {last_path}")
    fitness = None if trainer.fitness is None else float(trainer.fitness)
    best_fitness = None if trainer.best_fitness is None else float(trainer.best_fitness)
    is_best = fitness is not None and best_fitness is not None and fitness == best_fitness
    if is_best:
        wb_state["best_epoch"] = epoch
    artifact = wandb.Artifact(
        name=f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints",
        type="model",
        description="Checkpoint penuh per epoch: last.pt, best-to-date.pt, results.csv, args.yaml",
        metadata={
            "epoch": epoch, "fitness": fitness, "best_fitness": best_fitness,
            "is_best_epoch": bool(is_best), "best_epoch_so_far": wb_state["best_epoch"],
            "experiment": EXPERIMENT_ID, "dataset": DATA, "seed": SEED,
            "dataset_provenance": DATASET_PROVENANCE, "environment": ENVIRONMENT_INFO,
        },
    )
    artifact.add_file(str(last_path), name="last.pt")
    if best_path.is_file():
        artifact.add_file(str(best_path), name="best.pt")
    for filename in ("results.csv", "args.yaml"):
        file_path = Path(trainer.save_dir) / filename
        if file_path.is_file():
            artifact.add_file(str(file_path), name=filename)
    aliases = [f"epoch-{epoch:03d}", "latest"]
    if is_best:
        aliases.append("best")
    # Synchronous wait menjamin isi artifact epoch ini selesai dikirim sebelum file lokal berubah.
    wb_run.log_artifact(artifact, aliases=aliases).wait()
    wb_state["uploaded_epochs"].add(epoch)
    print(f"W&B checkpoint: epoch={epoch}, best={is_best}, aliases={aliases}")

def log_epoch_metrics(trainer):
    """Satu record grafik per epoch (training loss, KD loss, validation, LR, best epoch)."""
    epoch = int(trainer.epoch) + 1
    # Ultralytics memanggil on_fit_epoch_end sekali lagi pada final_eval; abaikan panggilan ekstra.
    if epoch not in wb_state["uploaded_epochs"] or epoch in wb_state["logged_epochs"]:
        return
    loss = trainer.label_loss_items(trainer.tloss, prefix="train")
    payload = {"epoch": epoch, **{k: float(v) for k, v in loss.items()},
               **{k: float(v) for k, v in trainer.lr.items()}}
    for key, value in trainer.metrics.items():
        try:
            payload[key] = float(value)
        except (ValueError, TypeError):
            continue
    payload["checkpoint/is_best_epoch"] = int(wb_state["best_epoch"] == epoch)
    payload["checkpoint/best_epoch_so_far"] = wb_state["best_epoch"] or 0
    wb_run.log(payload, step=epoch)
    wb_state["logged_epochs"].add(epoch)

def run_full_training():
    model = None
    with exclusive_gpu():
        try:
            init_seeds(SEED, deterministic=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(DEVICE)
            model = YOLO(MODEL_WEIGHTS)
            model.add_callback("on_train_start", bind_seed_callback(strict_training_protocol))
            model.add_callback("on_train_batch_start", bind_seed_callback(strict_batch_start))
            model.add_callback("on_train_batch_end", bind_seed_callback(strict_batch_end))
            train_args = dict(  # noqa: C408 -- Keep the source baseline argument block.
                data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
                weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
                project=str(SESSION_DIR), name="training", exist_ok=False,
                workers=WORKERS, patience=PATIENCE, plots=True,
                amp=AMP, pretrained=True, save=True, compile=False, resume=False,
            )
            train_args.update(KD_TRAIN_ARGS)
            wb_run.config.update({"training_args": train_args, "kd": KD_CONFIG})
            model.add_callback("on_model_save", bind_seed_callback(save_epoch_artifact))
            model.add_callback("on_fit_epoch_end", bind_seed_callback(log_epoch_metrics))
            start = time.perf_counter()
            model.train(**train_args)
            directory = Path(model.trainer.save_dir)
            best = directory / "weights/best.pt"
            if not best.is_file():
                raise FileNotFoundError(f"Checkpoint best.pt tidak ditemukan: {best}")
            return {"run_dir": str(directory), "best_pt": str(best),
                    "last_pt": str(directory / "weights/last.pt"),
                    "training_seconds": time.perf_counter() - start,
                    "completed_epochs": len(wb_state["uploaded_epochs"]), "best_epoch": wb_state["best_epoch"]}
        finally:
            model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


def validate_seed_list(seeds):
    """Reject ambiguous queues before opening any run or creating a model."""
    if not isinstance(seeds, (list, tuple)) or not seeds:
        raise ValueError("SEEDS_TO_RUN harus berupa daftar seed yang tidak kosong.")
    if any(isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32 for seed in seeds):
        raise ValueError("Setiap seed harus integer dalam rentang 0 sampai 2**32-1.")
    if len(seeds) != len(set(seeds)):
        raise ValueError("SEEDS_TO_RUN tidak boleh memiliki seed duplikat.")
    return list(seeds)


def seed_run_name(seed):
    return f"baseline_yolo26n_amd_seed{seed}"


def bind_seed_callback(callback):
    """Bind the active seed/run/state; reject stale callbacks instead of mixing logs."""
    seed, run, state = SEED, wb_run, wb_state
    def bound(trainer):
        if SEED != seed or wb_run is not run or wb_state is not state:
            raise RuntimeError("Callback berasal dari seed/run lain; logging dihentikan.")
        return callback(trainer)
    return bound


def start_seed_run():
    global wb_run
    NEW_WANDB_RUN_ID = wandb.util.generate_id()
    log_status("W&B", f"Membuat run baru: name={WANDB_RUN_NAME}, id={NEW_WANDB_RUN_ID}, resume=never...")
    diagnostic = DATA == "coco8.yaml"
    wb_run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=WANDB_RUN_NAME,
        id=NEW_WANDB_RUN_ID, resume="never", save_code=False, settings=wandb.Settings(init_timeout=90, disable_code=True),
        group=f"diagnostic-{WANDB_GROUP}" if diagnostic else WANDB_GROUP, job_type="training",
        tags=[*WANDB_TAGS, "diagnostic" if diagnostic else "research"],
        config={"experiment": EXPERIMENT_ID, "method": METHOD, "model": MODEL_WEIGHTS,
                "dataset": DATA, "eval_split": EVAL_SPLIT, "seed": SEED,
                "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "device": str(DEVICE),
                "workers": WORKERS, "optimizer": OPTIMIZER, "learning_rate": LR,
                "patience": PATIENCE, "amp": AMP, "weight_decay": WEIGHT_DECAY,
                "warmup_epochs": WARMUP_EPOCHS, "close_mosaic": CLOSE_MOSAIC,
                "teacher_artifact": None,
                "environment": ENVIRONMENT_INFO, "dataset_provenance": DATASET_PROVENANCE, "seeds_to_run": list(SEEDS_TO_RUN)})
    log_status("W&B", f"Run aktif: {wb_run.url}")


def run_one_seed(seed, queue_dir):
    """Complete one independent run, including evaluation/upload/finish, before returning."""
    global SEED, SESSION_DIR, WANDB_RUN_NAME, WANDB_TAGS, wb_run, wb_state
    if wandb.run is not None:
        raise RuntimeError("Masih ada run W&B aktif; seed baru tidak boleh dimulai.")
    SEED = seed
    WANDB_RUN_NAME = seed_run_name(seed)
    WANDB_TAGS = [*BASE_WANDB_TAGS, f"seed{seed}"]
    SESSION_DIR = Path(queue_dir) / f"seed-{seed}"
    SESSION_DIR.mkdir(parents=True, exist_ok=False)
    wb_run = None
    wb_state = {"saved_epochs": set(), "uploaded_epochs": set(), "logged_epochs": set(),
                "best_epoch": None, "epoch_upload_seconds": []}
    try:
        init_seeds(seed, deterministic=True)
        start_seed_run()
        prepare_seed_teacher()
        log_status("SEED", f"Memulai seed {seed}: {EPOCHS} epoch, student baru dari {MODEL_WEIGHTS}.")
        training = run_full_training()
        (SESSION_DIR / "training_result.json").write_text(json.dumps(training, indent=2), encoding="utf-8")
        with exclusive_gpu():
            result = evaluate_final_training(training)
        result.update(seed=seed, wandb_run_id=wb_run.id, wandb_run_name=WANDB_RUN_NAME,
                      wandb_url=wb_run.url, seed_directory=str(SESSION_DIR))
        (SESSION_DIR / "seed_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        # finish() waits for the run's remaining W&B sync; the next seed starts afterwards.
        wb_run.finish()
        log_status("SEED", f"Seed {seed} selesai; run W&B sudah ditutup.")
        return result
    except BaseException:
        if wb_run is not None:
            try:
                wb_run.finish(exit_code=1)
            except Exception as finish_error:  # noqa: BLE001 -- Close the failed run without masking its original error.
                log_status("W&B", f"Penutupan run gagal ({type(finish_error).__name__}); antrean dihentikan.")
        raise
    finally:
        wb_run = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_seed_queue():
    """Serial queue: persist completed seeds and stop immediately on any failure."""
    validate_api_keys()
    require_rocm_runtime()
    if DATA is None or not isinstance(globals().get("DATASET_PROVENANCE"), dict):
        raise RuntimeError("Jalankan cell dataset Roboflow sebelum antrean seed.")
    if hashlib.sha256(Path(DATA).read_bytes()).hexdigest() != DATASET_PROVENANCE["yaml_sha256"]:
        raise RuntimeError("YAML berubah setelah validasi; jalankan kembali cell dataset.")
    seeds = validate_seed_list(SEEDS_TO_RUN)
    if not WANDB_ENABLED:
        raise ValueError("Antrean penelitian ini memerlukan W&B online.")
    if wandb.run is not None:
        raise RuntimeError("Selesaikan run W&B aktif sebelum menjalankan antrean ini.")
    token = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")  # noqa: UP017 -- Source baseline timestamp.
    queue_dir = Path(PROJECT) / EXPERIMENT_ID / f"multiseed_{token}"
    queue_dir.mkdir(parents=True, exist_ok=False)
    results = []
    progress = {"experiment": EXPERIMENT_ID, "seeds_to_run": seeds, "completed": [],
                "failed_seed": None, "status": "running"}
    progress_path = queue_dir / "multiseed_progress.json"
    progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    log_status("QUEUE", f"Seed serial {seeds}; hasil: {queue_dir}. Seed yang dikomentari tidak ditambahkan otomatis.")
    for seed in seeds:
        try:
            result = run_one_seed(seed, queue_dir)
            results.append(result)
            (queue_dir / "multiseed_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
            progress["completed"].append({"seed": seed, "wandb_run_id": result["wandb_run_id"],
                                          "seed_directory": result["seed_directory"]})
            progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
        except BaseException:
            progress.update(failed_seed=seed, status="failed")
            progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
            log_status("QUEUE", f"Berhenti pada seed {seed}. Hasil seed selesai tetap tersimpan; untuk lanjut isi SEEDS_TO_RUN dengan seed yang belum selesai.")
            raise
    progress["status"] = "completed"
    progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    log_status("QUEUE", f"Selesai: {len(results)} run terpisah. Gabungkan hasil 42 yang sudah ada saat pelaporan.")
    return results


## 7. Evaluasi best checkpoint

Evaluasi student-only best.pt pada val, lalu unggah metrik/config/grafik sebagai results artifact. Metadata mencatat backend ROCm dan sumber dataset; waktu inferensi/performa antarhardware tidak dianggap identik. Snapshot notebook dan API key tidak diunggah.


In [ ]:
def evaluate_final_training(training):
    best_model = metrics = None
    try:
        run_dir = Path(training["run_dir"])
        best_pt = Path(training["best_pt"])
        training_seconds = training["training_seconds"]
        # Evaluasi dari checkpoint terbaik (bukan last.pt).
        best_model = YOLO(str(best_pt))
        metrics = best_model.val(
            data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH,
            device=DEVICE, plots=True, verbose=False,
        )

        from ultralytics.utils.torch_utils import get_flops
        parameters = sum(p.numel() for p in best_model.model.parameters())
        try:
            gflops = float(get_flops(best_model.model, imgsz=IMGSZ))
        except (AttributeError, ImportError, RuntimeError, TypeError, ValueError) as error:
            print("GFLOPs tidak tersedia:", error)
            gflops = None

        results = {
            "experiment": EXPERIMENT_ID,
            "method": METHOD,
            "model": MODEL_WEIGHTS,
            "dataset": DATA,
            "dataset_provenance": DATASET_PROVENANCE,
            "environment": ENVIRONMENT_INFO,
            "eval_split": EVAL_SPLIT,
            "seed": SEED,
            "epochs": EPOCHS,
            "imgsz": IMGSZ,
            "batch": BATCH,
            "optimizer": OPTIMIZER,
            "learning_rate": LR,
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "map50": float(metrics.box.map50),
            "map50_95": float(metrics.box.map),
            "parameters": int(parameters),
            "gflops": gflops,
            "inference_ms_per_image": float(metrics.speed.get("inference", 0.0)),
            "training_seconds": float(training_seconds),
            "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "teacher_checkpoint": str(TEACHER_CHECKPOINT) if METHOD != "none" else None,
            "best_pt": str(best_pt),
        }
        if torch.cuda.is_available():
            results["peak_gpu_memory_gb"] = float(torch.cuda.max_memory_allocated() / 1024**3)
        print(json.dumps(results, indent=2, ensure_ascii=False))
        # File lokal tetap tersimpan oleh Ultralytics, W&B menyimpan versi artifact tiap epoch.
        run_dir.mkdir(parents=True, exist_ok=True)
        results["best_epoch"] = wb_state["best_epoch"] if WANDB_ENABLED else None
        (run_dir / "metrics.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
        configuration = {
            "experiment": EXPERIMENT_ID, "dataset": DATA, "evaluation_split": EVAL_SPLIT,
            "model": MODEL_WEIGHTS, "teacher": TEACHER_CHECKPOINT if METHOD != "none" else None,
            "seed": SEED, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
            "optimizer": OPTIMIZER, "lr0": LR, "device": DEVICE,
            "workers": WORKERS, "patience": PATIENCE, "amp": AMP,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC,
            "kd_hyperparameters": KD_CONFIG,
            "dataset_provenance": DATASET_PROVENANCE, "environment": ENVIRONMENT_INFO,
            "wandb_enabled": WANDB_ENABLED, "wandb_project": WANDB_PROJECT,
        }
        (run_dir / "config.json").write_text(json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8")

        if wb_run is not None:
            summary = {f"final/{key}": val for key, val in results.items()
                       if isinstance(val, (int, float)) and not isinstance(val, bool) and val is not None}
            wb_run.log(summary)
            wb_run.summary.update(summary)
            wb_run.summary["best_epoch"] = wb_state["best_epoch"]
            wb_run.summary["checkpoint_collection"] = f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints"
            wb_run.summary["uploaded_checkpoint_epochs"] = len(wb_state["uploaded_epochs"])

            # Ringkasan/config dan grafik sebagai Artifact terpisah, tanpa menggandakan checkpoint model.
            final_artifact = wandb.Artifact(
                name=f"{EXPERIMENT_ID}-{wb_run.id}-results", type="results",
                metadata={"experiment": EXPERIMENT_ID, "seed": SEED, "best_epoch": wb_state["best_epoch"],
                          "dataset_provenance": DATASET_PROVENANCE, "environment": ENVIRONMENT_INFO},
            )
            for filename in ("metrics.json", "config.json", "results.csv", "args.yaml"):
                candidate = run_dir / filename
                if candidate.is_file():
                    final_artifact.add_file(str(candidate), name=filename)
            for folder_name, folder in (("training", run_dir),
                                        ("evaluation", Path(getattr(metrics, "save_dir", run_dir)))):
                if folder.exists():
                    for plot_path in sorted(folder.glob("*.png")):
                        key = f"plots/{folder_name}/{plot_path.stem}"
                        wb_run.log({key: wandb.Image(str(plot_path))})
                        final_artifact.add_file(str(plot_path), name=f"{folder_name}/{plot_path.name}")
            wb_run.log_artifact(final_artifact, aliases=["final"]).wait()
            print("W&B:", wb_run.url)
            print("Checkpoint alias: epoch-001 ... epoch-%03d, latest, best" % len(wb_state["uploaded_epochs"]))  # noqa: UP031 -- Source baseline output.

        print("Hasil lokal:", run_dir)
        print("  - weights/best.pt, weights/last.pt")
        print("  - results.csv, args.yaml")
        print("  - metrics.json, config.json")

        return results

    finally:
        best_model = metrics = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## 8. Jalankan training dan evaluasi serial

Run All menjalankan seluruh seed aktif setelah pemeriksaan lingkungan/dataset. Setiap seed ditutup sebelum seed berikutnya. Jika sesi cloud terputus, checkpoint epoch yang berhasil diunggah tersedia di W&B; tidak ada klaim dukungan resume engine atau agregasi mean/std otomatis.


In [ ]:
MULTISEED_RESULTS = run_seed_queue()
print(json.dumps(MULTISEED_RESULTS, indent=2))
